# CNN-Transformer HOSE Training on Colab

This notebook clones the repo, loads the pre-sharded training data prepared for GitHub, trains the 3-class CNN-Transformer (`Avoid`/`Hold`/`Buy`) on GPU if available, evaluates the best checkpoint with batched test inference, exports weekly top-5 picks for the v2 full-reset strategy, and plots test-period portfolio performance versus VNINDEX.

Pinned configuration for this notebook:
- Labels = v2 baseline `30/40/30` (`Buy` top 30%, `Hold` middle 40%, `Avoid` bottom 30%)
- Feature packs = `base`
- Model candidate = `model_exp_01_attention_pooling`
- Weekly execution = top-5 by score `P(Buy) - P(Avoid)`, full reset each week
- Use this notebook only with `data/colab` rebuilt from the same feature profile


In [ ]:
!nvidia-smi || true

In [ ]:
%cd /content
!rm -rf trading__weekly_CNN_Transformer

REPO_URL = 'https://github.com/votranhuonggiang/trading__weekly_CNN_Transformer.git'
REPO_BRANCH = 'experiment/model-exp-01-attention-pooling-base-only-colab'

!git clone -b {REPO_BRANCH} {REPO_URL}

import os
if not os.path.isdir('/content/trading__weekly_CNN_Transformer'):
    raise RuntimeError(
        f'Clone failed for branch {REPO_BRANCH}. Push that branch to GitHub first, ' 
        'or change REPO_BRANCH to an existing branch before continuing.'
    )

%cd /content/trading__weekly_CNN_Transformer
!pip install -q -r requirements.txt
!git branch --show-current
!git log --oneline -n 3
!grep -n "build_top_k_portfolio" src/predict.py


In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('seed:', SEED)
print('cudnn_deterministic:', torch.backends.cudnn.deterministic)
print('cudnn_benchmark:', torch.backends.cudnn.benchmark)


In [ ]:
import os
import sys

REPO_ROOT = '/content/trading__weekly_CNN_Transformer'
if not os.path.isdir(REPO_ROOT):
    raise RuntimeError(
        'Repo folder not found at /content/trading__weekly_CNN_Transformer. ' 
        'Run the clone cell above first and make sure it succeeds.'
    )

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]

from src.colab_data import load_feature_profile, load_sharded_arrays
from src.train import TrainingConfig, train_model
from src.model import CNNTransformerClassifier
from src.predict import build_portfolio_performance, build_top_k_portfolio, build_weekly_prediction_table, export_prediction_outputs
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

arrays = load_sharded_arrays('data/colab')
metadata = pd.read_parquet('data/colab/model_dataset_metadata.parquet')
profile = load_feature_profile('data/colab')
vnindex_comparison = pd.read_csv('outputs/tables/vnindex_label_comparison.csv')

expected_metadata_cols = {
    'rebalance_date',
    'next_rebalance_date',
    'ticker',
    'label',
    'label_name',
    'next_week_return',
    'return_rank_pct',
    'split',
}
expected_label_names = {'Avoid', 'Hold', 'Buy'}
expected_feature_packs = ['base']
expected_label_mode = 'v2_buy_top30_hold_mid40_avoid_bottom30'
actual_label_names = set(metadata['label_name'].dropna().unique())
if not expected_metadata_cols.issubset(metadata.columns) or actual_label_names != expected_label_names:
    raise RuntimeError(
        'This notebook is pinned to v2 baseline labels (30/40/30) and requires 3-class labels (Avoid/Hold/Buy). ' 
        'Rebuild data/colab with the selected feature profile before rerunning. ' 
        f'Current metadata columns: {metadata.columns.tolist()}; labels: {sorted(actual_label_names)}'
    )
if profile.get('feature_packs') != expected_feature_packs:
    raise RuntimeError(
        'This notebook is pinned to feature pack base only. ' 
        f"Found feature_packs={profile.get('feature_packs')} in data/colab/feature_profile.json"
    )
if profile.get('label_mode') != expected_label_mode:
    raise RuntimeError(
        'This notebook is pinned to v2 baseline labels 30/40/30. ' 
        f"Found label_mode={profile.get('label_mode')} in data/colab/feature_profile.json"
    )
if int(profile.get('num_features', -1)) != int(arrays['X_train'].shape[-1]):
    raise RuntimeError(
        'Feature profile num_features does not match array width. ' 
        f"profile={profile.get('num_features')} arrays={arrays['X_train'].shape[-1]}"
    )

class_names = ['Avoid', 'Hold', 'Buy']

metadata['rebalance_date'] = pd.to_datetime(metadata['rebalance_date']).dt.normalize()
metadata['next_rebalance_date'] = pd.to_datetime(metadata['next_rebalance_date']).dt.normalize()
vnindex_comparison['rebalance_date'] = pd.to_datetime(vnindex_comparison['rebalance_date']).dt.normalize()
vnindex_comparison['next_rebalance_date'] = pd.to_datetime(vnindex_comparison['next_rebalance_date']).dt.normalize()

print('cwd:', os.getcwd())
print('sys.path[0]:', sys.path[0])
print({k: v.shape for k, v in arrays.items() if hasattr(v, 'shape')})
print('metadata_rows:', len(metadata))
print('metadata_cols:', metadata.columns.tolist())
print('label_mode:', profile['label_mode'])
print('feature_packs:', profile['feature_packs'])
print('num_features:', profile['num_features'])
print('profile_name:', profile.get('profile_name', 'n/a'))
print('model_candidate: model_exp_01_attention_pooling')
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))


In [ ]:
config = TrainingConfig(
    epochs=15,
    batch_size=512 if torch.cuda.is_available() else 128,
    learning_rate=1e-3,
    patience=4,
    checkpoint_dir='checkpoints'
)
model, metrics = train_model(arrays, config)
print('best_epoch:', metrics['best_epoch'])
print('accuracy:', metrics['accuracy'])
print('macro_f1:', metrics['macro_f1'])
print(metrics['report'])
print('checkpoint:', metrics['checkpoint_path'])

In [ ]:
history = metrics['history']
epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_accuracy = [row['val_accuracy'] for row in history]
val_macro_f1 = [row['val_macro_f1'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, marker='o', color='#1f4e79')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, val_accuracy, marker='o', label='Val Accuracy', color='#117a65')
axes[1].plot(epochs, val_macro_f1, marker='o', label='Val Macro F1', color='#b03a2e')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
device = torch.device(config.device)
num_classes = int(arrays['y_train'].max()) + 1
if num_classes != 3:
    raise RuntimeError(
        f'This notebook is pinned to architecture_v2.md and expects a 3-class Avoid/Hold/Buy model head. Found num_classes={num_classes}.'
    )
class_names = ['Avoid', 'Hold', 'Buy']
best_model = CNNTransformerClassifier(num_features=arrays['X_train'].shape[-1], num_classes=num_classes).to(device)
best_model.load_state_dict(torch.load(metrics['checkpoint_path'], map_location=device))
best_model.eval()

test_batch_size = 1024 if torch.cuda.is_available() else 256
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(arrays['X_test'], dtype=torch.float32),
        torch.tensor(arrays['y_test'], dtype=torch.long),
    ),
    batch_size=test_batch_size,
    shuffle=False,
    num_workers=0,
)

preds = []
probs_list = []
targets = []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        logits = best_model(x_batch.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred = np.argmax(probs, axis=1)
        probs_list.append(probs)
        preds.append(pred)
        targets.append(y_batch.numpy())

probs = np.concatenate(probs_list)
if probs.ndim != 2 or probs.shape[1] != 3:
    raise RuntimeError(f'Expected probability array with shape [N, 3] for Avoid/Hold/Buy. Got shape={probs.shape}.')
preds = np.concatenate(preds)
y_test = np.concatenate(targets)

print('test_accuracy:', accuracy_score(y_test, preds))
print('test_macro_f1:', f1_score(y_test, preds, average='macro'))
print('predicted_class_counts:', dict(zip(class_names, np.bincount(preds, minlength=num_classes).tolist())))
print(classification_report(y_test, preds, target_names=class_names, digits=4, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Test Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
avg_probs = probs.mean(axis=0)
bar_colors = ['#b03a2e', '#7f8c8d', '#117a65']
plt.figure(figsize=(7, 4))
plt.bar(class_names, avg_probs, color=bar_colors)
plt.title('Average Predicted Probability on Test Set')
plt.ylabel('Probability')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
EXPERIMENT_TAG = 'model_exp_01_attention_pooling_base_only'
TOP_K = 5
OUTPUT_DIR = f'outputs/predictions/{EXPERIMENT_TAG}'

all_x = np.concatenate([arrays['X_train'], arrays['X_val'], arrays['X_test']], axis=0)
all_probs = []
all_loader = DataLoader(
    TensorDataset(torch.tensor(all_x, dtype=torch.float32)),
    batch_size=1024 if torch.cuda.is_available() else 256,
    shuffle=False,
    num_workers=0,
)

with torch.no_grad():
    for (x_batch,) in all_loader:
        logits = best_model(x_batch.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

all_probs = np.concatenate(all_probs)
if all_probs.ndim != 2 or all_probs.shape[1] != 3:
    raise RuntimeError(f'Expected batched inference probabilities with shape [N, 3] for Avoid/Hold/Buy. Got shape={all_probs.shape}.')
prediction_table = build_weekly_prediction_table(metadata, all_probs)

# Model candidate 01: attention pooling on top of the base-only v2 baseline.
top_table = build_top_k_portfolio(prediction_table, top_k=TOP_K)
top_table['rebalance_date'] = pd.to_datetime(top_table['rebalance_date']).dt.normalize()
os.makedirs(OUTPUT_DIR, exist_ok=True)
score_path, top_path = export_prediction_outputs(prediction_table, OUTPUT_DIR, top_k=TOP_K, buy_only=False)

fee_rates = [0.0, 0.0015, 0.0025, 0.0035]
summary_rows = []
perf_tables = []
for fee_rate in fee_rates:
    perf = build_portfolio_performance(top_table, vnindex_comparison=vnindex_comparison, fee_rate=fee_rate)
    perf['rebalance_date'] = pd.to_datetime(perf['rebalance_date']).dt.normalize()
    perf['fee_rate'] = fee_rate
    perf_tables.append(perf)

    test_perf_fee = perf[perf['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
    if len(test_perf_fee) == 0:
        continue

    port_std = test_perf_fee['portfolio_simple_return'].std()
    port_sharpe = np.sqrt(52) * test_perf_fee['portfolio_simple_return'].mean() / port_std if port_std > 0 else np.nan

    vn_sharpe = np.nan
    if 'vnindex_simple_return' in test_perf_fee.columns and test_perf_fee['vnindex_simple_return'].notna().any():
        vn_std = test_perf_fee['vnindex_simple_return'].std()
        vn_sharpe = np.sqrt(52) * test_perf_fee['vnindex_simple_return'].mean() / vn_std if vn_std > 0 else np.nan

    summary_rows.append({
        'experiment_tag': EXPERIMENT_TAG,
        'top_k': TOP_K,
        'fee_rate': fee_rate,
        'weeks_test': int(len(test_perf_fee)),
        'avg_num_holdings_test': float(test_perf_fee['num_holdings'].mean()),
        'avg_cash_weight_test': float(test_perf_fee['cash_weight'].mean()),
        'portfolio_total_return_test': float((1.0 + test_perf_fee['portfolio_simple_return']).prod() - 1.0),
        'portfolio_sharpe_test': float(port_sharpe) if pd.notna(port_sharpe) else np.nan,
        'vnindex_sharpe_test': float(vn_sharpe) if pd.notna(vn_sharpe) else np.nan,
        'avg_turnover_test': float(test_perf_fee['portfolio_turnover'].mean()) if 'portfolio_turnover' in test_perf_fee.columns else np.nan,
        'avg_trading_cost_test': float(test_perf_fee['trading_cost'].mean()) if 'trading_cost' in test_perf_fee.columns else np.nan,
    })

perf_all = pd.concat(perf_tables, ignore_index=True)
summary_df = pd.DataFrame(summary_rows).sort_values('fee_rate').reset_index(drop=True)

base_fee_rate = 0.0025
perf_base = perf_all[perf_all['fee_rate'] == base_fee_rate].copy()
performance_csv = f'{OUTPUT_DIR}/weekly_top_{TOP_K}_model_exp_01_attention_pooling_base_only_full_reset_performance.csv'
summary_csv = f'{OUTPUT_DIR}/weekly_top_{TOP_K}_model_exp_01_attention_pooling_base_only_full_reset_cost_sensitivity.csv'
perf_base.to_csv(performance_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

test_perf = perf_base[perf_base['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
test_perf['portfolio_cumulative_test'] = (1.0 + test_perf['portfolio_simple_return']).cumprod()
if 'vnindex_simple_return' in test_perf.columns and test_perf['vnindex_simple_return'].notna().any():
    test_perf['vnindex_cumulative_test'] = (1.0 + test_perf['vnindex_simple_return']).cumprod()

print('experiment_tag:', EXPERIMENT_TAG)
print('score_csv:', score_path)
print('top_csv:', top_path)
print('performance_csv:', performance_csv)
print('cost_sensitivity_csv:', summary_csv)
print('base_fee_rate:', base_fee_rate)
print('execution: weekly full reset, buy top-5 next week by score = P(Buy) - P(Avoid)')
print('feature_profile:', profile)
display(summary_df)
display(test_perf.head(10))


In [ ]:
# Optional binary Buy/NotBuy diagnostic derived from the 3-class v2 model.
# This does NOT change the exported top-5 portfolio, which is always ranked by score = P(Buy) - P(Avoid).
bn_table = build_weekly_prediction_table(metadata, all_probs).copy()
bn_table['rebalance_date'] = pd.to_datetime(bn_table['rebalance_date']).dt.normalize()

print('split value counts:', bn_table['split'].value_counts(dropna=False).to_dict())

y_true_bin = (bn_table['label_name'] == 'Buy').astype(int).to_numpy()
p_buy = bn_table['p_buy'].to_numpy()

val_mask = bn_table['split'].isin(['validation', 'val']).to_numpy()
test_mask = bn_table['split'].eq('test').to_numpy()

if val_mask.sum() == 0:
    raise ValueError(
        f"No validation rows found in split column. Found splits: {sorted(bn_table['split'].dropna().unique().tolist())}"
    )
if test_mask.sum() == 0:
    raise ValueError(
        f"No test rows found in split column. Found splits: {sorted(bn_table['split'].dropna().unique().tolist())}"
    )

thresholds = np.arange(0.30, 0.71, 0.02)
rows = []
for th in thresholds:
    y_pred_val = (p_buy[val_mask] >= th).astype(int)
    tp = int(((y_true_bin[val_mask] == 1) & (y_pred_val == 1)).sum())
    pred_pos = int((y_pred_val == 1).sum())
    actual_pos = int((y_true_bin[val_mask] == 1).sum())

    precision = tp / pred_pos if pred_pos > 0 else 0.0
    recall = tp / actual_pos if actual_pos > 0 else 0.0

    rows.append({
        'threshold': float(th),
        'val_accuracy': float(accuracy_score(y_true_bin[val_mask], y_pred_val)),
        'val_buy_precision': float(precision),
        'val_buy_recall': float(recall),
        'val_buy_f1': float(f1_score(y_true_bin[val_mask], y_pred_val, zero_division=0)),
    })

bn_thresholds = pd.DataFrame(rows).sort_values('val_buy_f1', ascending=False).reset_index(drop=True)
best_threshold = float(bn_thresholds.loc[0, 'threshold'])
print('best_threshold_by_val_buy_f1:', best_threshold)
display(bn_thresholds.head(10))

y_pred_test = (p_buy[test_mask] >= best_threshold).astype(int)
print('binary_test_accuracy_at_threshold:', accuracy_score(y_true_bin[test_mask], y_pred_test))
print('binary_test_buy_f1_at_threshold:', f1_score(y_true_bin[test_mask], y_pred_test, zero_division=0))
print(classification_report(
    y_true_bin[test_mask],
    y_pred_test,
    target_names=['NotBuy', 'Buy'],
    digits=4,
    zero_division=0,
))

cm_bn = confusion_matrix(y_true_bin[test_mask], y_pred_test)
disp_bn = ConfusionMatrixDisplay(confusion_matrix=cm_bn, display_labels=['NotBuy', 'Buy'])
fig, ax = plt.subplots(figsize=(5, 5))
disp_bn.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Binary Buy/NotBuy Diagnostic (threshold={best_threshold:.2f})')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

required_cols = {'rebalance_date', 'portfolio_cumulative_test'}
missing = [c for c in required_cols if c not in test_perf.columns]
if missing:
    raise ValueError(f"test_perf is missing required columns: {missing}. Available: {test_perf.columns.tolist()}")

plot_df = test_perf.copy()
plot_df['rebalance_date'] = pd.to_datetime(plot_df['rebalance_date'], errors='coerce')
plot_df = plot_df.dropna(subset=['rebalance_date', 'portfolio_cumulative_test']).sort_values('rebalance_date')

if plot_df.empty:
    raise ValueError('No rows to plot after cleaning test_perf. Check upstream filtering and cumulative calculations.')

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(
    plot_df['rebalance_date'],
    plot_df['portfolio_cumulative_test'],
    label='Top 5 Weekly Reset Portfolio (attention pooling + base only, Test)',
    color='#117a65',
    linewidth=2.2,
)

if 'vnindex_cumulative_test' in plot_df.columns:
    vn = plot_df.dropna(subset=['vnindex_cumulative_test'])
    if not vn.empty:
        ax.plot(
            vn['rebalance_date'],
            vn['vnindex_cumulative_test'],
            label='VNINDEX (Test)',
            color='#1f4e79',
            linewidth=2.0,
        )

ax.set_title('Top 5 Weekly Reset Portfolio With attention pooling and base-only features vs VNINDEX - Test Period Only')
ax.set_xlabel('Rebalance Date')
ax.set_ylabel('Cumulative Growth of 1.0')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import os
import shutil
from google.colab import files

candidates = [
    'outputs/predictions/model_exp_01_attention_pooling_base_only',
    'outputs/predictions/only_buy',
    'outputs/predictions',
]
source_dir = next((d for d in candidates if os.path.isdir(d)), None)

if source_dir is None:
    raise FileNotFoundError(
        'No prediction folder found. Run export/backtest cells first. ' 
        'Checked: outputs/predictions/model_exp_01_attention_pooling_base_only, outputs/predictions/only_buy, outputs/predictions'
    )

if source_dir.endswith('full_v2_base_cross_sectional_liquidity'):
    zip_base = 'model_exp_01_attention_pooling_base_only_predictions'
elif source_dir.endswith('only_buy'):
    zip_base = 'only_buy_predictions'
else:
    zip_base = 'predictions_export'

zip_path = shutil.make_archive(zip_base, 'zip', source_dir)
print(f'Zipped from: {source_dir}')
print(f'Created: {zip_path}')
files.download(zip_path)
